---
title: Ordinary Least Squares
---


::: {note} Learning Outcomes
* Define linearity with respect to a vector of parameters $\theta$.
* Understand the use of matrix notation to express multiple linear regression.
* Interpret ordinary least squares as the minimization of the norm of the residual vector.
* Compute performance metrics for multiple linear regression.
* Use `scikit-learn` to fit models and make predictions.
:::

We've now spent a number of lectures exploring how to build effective models – we introduced the SLR and constant models, selected cost functions to suit our modeling task, and applied transformations to improve the linear fit.

Throughout all of this, we considered models of one feature ($\hat{y}_i = \theta_0 + \theta_1 x_i$) or zero features ($\hat{y}_i = \theta_0$). As data scientists, we usually have access to datasets containing *many* features. To make the best models we can, it will be beneficial to consider all of the variables available to us as inputs to a model, rather than just one. In today's lecture, we'll introduce **multiple linear regression** as a framework to incorporate multiple features into a model. We will also learn how to accelerate the modeling process – specifically, we'll see how linear algebra offers us a powerful set of tools for understanding model performance.

## OLS Problem Formulation

### Multiple Linear Regression

Multiple linear regression is an extension of simple linear regression that adds additional features to the model. The multiple linear regression model takes the form:

$$
\hat{y} = \theta_0\:+\:\theta_1x_{1}\:+\:\theta_2 x_{2}\:+\:...\:+\:\theta_p x_{p}
$$

Our predicted value of $y$, $\hat{y}$, is a linear combination of the single **observations** (features), $x_i$, and the parameters, $\theta_i$.

$$
\vec{x} = \begin{bmatrix} x_1 \\ \vdots \\ x_p \end{bmatrix} \quad\rightarrow\quad \begin{bmatrix} \theta_0 \\ \theta_1 \\ \vdots \\ \theta_p \end{bmatrix} \quad\rightarrow\quad \hat{y}
$$

As an example, you could predict cats' ages $\hat{y}$ as a linear model of two features: length $x_1$ and weight $x_2$. This would give us the equation

$$
\hat{y} = \theta_0 + \theta_1x_1 + \theta_2x_2
$$

where $\theta_0$ is the intercept, $\theta_1$ is the parameter for length, and $\theta_2$ is the parameter for weight.

We can explore this idea further by looking at a dataset containing aggregate per-player data from the 2018-19 NBA season, downloaded from [Kaggle](https://www.kaggle.com/schmadam97/nba-regular-season-stats-20182019).

In [39]:
import polars as pl
import seaborn as sns
import numpy as np

nba = pl.read_csv('data/nba18-19.csv')

In [40]:
nba.head(5)

Rk,Player,Pos,Age,Tm,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS
i64,str,str,i64,str,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,"""Álex Abrines\abrinal01""","""SG""",25,"""OKC""",31,2,19.0,1.8,5.1,0.357,1.3,4.1,0.323,0.5,1.0,0.5,0.487,0.4,0.4,0.923,0.2,1.4,1.5,0.6,0.5,0.2,0.5,1.7,5.3
2,"""Quincy Acy\acyqu01""","""PF""",28,"""PHO""",10,0,12.3,0.4,1.8,0.222,0.2,1.5,0.133,0.2,0.3,0.667,0.278,0.7,1.0,0.7,0.3,2.2,2.5,0.8,0.1,0.4,0.4,2.4,1.7
3,"""Jaylen Adams\adamsja01""","""PG""",22,"""ATL""",34,1,12.6,1.1,3.2,0.345,0.7,2.2,0.338,0.4,1.1,0.361,0.459,0.2,0.3,0.778,0.3,1.4,1.8,1.9,0.4,0.1,0.8,1.3,3.2
4,"""Steven Adams\adamsst01""","""C""",25,"""OKC""",80,80,33.4,6.0,10.1,0.595,0.0,0.0,0.0,6.0,10.1,0.596,0.595,1.8,3.7,0.5,4.9,4.6,9.5,1.6,1.5,1.0,1.7,2.6,13.9
5,"""Bam Adebayo\adebaba01""","""C""",21,"""MIA""",82,28,23.3,3.4,5.9,0.576,0.0,0.2,0.2,3.4,5.7,0.588,0.579,2.0,2.8,0.735,2.0,5.3,7.3,2.2,0.9,0.8,1.5,2.5,8.9


Let's say we are interested in predicting the number of points (`PTS`) an athlete will score in a basketball game this season.

Suppose we want to fit a linear model by using some characteristics, or **features** of a player. Specifically, we'll focus on field goals, assists, and 3-point attempts.

* `FG`, the average number of (2-point) field goals per game
* `AST`, the average number of assists per game
* `3PA`, the average number of 3-point field goals attempted per game

In [41]:
nba.select(pl.col('FG'), pl.col('AST'), pl.col('3PA'), pl.col('PTS')).head()

FG,AST,3PA,PTS
f64,f64,f64,f64
1.8,0.6,4.1,5.3
0.4,0.8,1.5,1.7
1.1,1.9,2.2,3.2
6.0,1.6,0.0,13.9
3.4,2.2,0.2,8.9


Because we are now dealing with many parameter values, we've collected them all into a **parameter vector** with dimensions $(p+1) \times 1$ to keep things tidy. Remember that $p$ represents the number of features we have (in this case, 3).

$$
\theta = \begin{bmatrix}
           \theta_{0} \\
           \theta_{1} \\
           \vdots \\
           \theta_{p}
         \end{bmatrix}
$$

We are working with two vectors here: a row vector representing the observed data, and a column vector containing the model parameters. The multiple linear regression model is **equivalent to the dot (scalar) product of the observation vector and parameter vector**. 

$$
\begin{aligned}
[1,\:x_{1},\:x_{2},\:x_{3},\:...,\:x_{p}] \theta \quad&=\quad [1,\:x_{1},\:x_{2},\:x_{3},\:...,\:x_{p}] \begin{bmatrix}
           \theta_{0} \\
           \theta_{1} \\
           \vdots \\
           \theta_{p}
         \end{bmatrix} \\
\\
&=\quad \theta_0\:+\:\theta_1x_{1}\:+\:\theta_2 x_{2}\:+\:...\:+\:\theta_p x_{p}
\end{aligned}
$$

Notice that we have inserted 1 as the first value in the observation vector. When the dot product is computed, this 1 will be multiplied with $\theta_0$ to give the intercept of the regression model. We call this 1 entry the **intercept** or **bias** term.

Given that we have three features here, we can express this model as:
$$\hat{y} = \theta_0\:+\:\theta_1x_{1}\:+\:\theta_2 x_{2}\:+\:\theta_3 x_{3}$$

Our features are represented by $x_1$ (`FG`), $x_2$ (`AST`), and $x_3$ (`3PA`) with each having correpsonding parameters, $\theta_1$, $\theta_2$, and  $\theta_3$.

In statistics, this model + loss is called **Ordinary Least Squares (OLS)**. The solution to OLS is the minimizing loss for parameters $\hat{\theta}$, also called the **least squares estimate**.

### From One Feature To Many Features

In an earlier chapter, we worked with the Simple Linear Regression (SLR) model $\hat{y} = \theta_0 + \theta_1x$, along the actual observations $y$. We can represent the dataset that corresponds to this model as follows:

$$
\begin{array}{c}
\begin{array}{cc} \,x  & \quad  \!\!\!y \end{array} \\
\boxed{
\begin{array}{c|c}
x_1 & y_1 \\
\hline
x_2 & y_2 \\
\hline
\vdots & \vdots \\
\hline
x_n & y_n
\end{array}
}
\end{array}
\quad\rightarrow\quad
\begin{array}{c}
\begin{array}{cc} \mathbf{FG}  &  \mathbf{PTS} \end{array} \\
\boxed{
\begin{array}{c|c}
1.8\,\, & \enspace5.3 \\
\hline
0.4\,\, & \enspace1.7 \\
\hline
1.1\,\, & \enspace3.2 \\
\hline
6.0\,\, & 13.9 \\
\hline
3.4\,\, & \enspace8.9 \\
\hline
\vdots & \vdots \\
\end{array}
}
\end{array}
$$

We have also worked with the Constant Model $\hat{y} = \theta$, along with the actual observations $y$. We can represent the dataset that corresponds to this model even more simply, as there is no feature data to consider:

$$
\begin{array}{c}
y \\
\boxed{
\begin{array}{c}
y_1 \\
\hline
y_2 \\
\hline
\vdots \\
\hline
y_n
\end{array}
}
\end{array}
\quad\rightarrow\quad
\begin{array}{c}
\mathbf{PTS} \\
\boxed{
\begin{array}{c}
\,\,5.3 \\
\hline
\,\,1.7 \\
\hline
\,\,3.2 \\
\hline
\!13.9 \\
\hline
\,\,8.9 \\
\hline
\vdots \\
\end{array}
}
\end{array}
$$

Accordingly, we can use a similar represenation for our new Multiple Regression Model:

$$
\begin{array}{c}
\begin{array}{ccccc} 
x_{:,1} & \enspace\!\!\! x_{:,2} & \quad & \,\, x_{:,p} & \,\, y
\end{array} \\
\boxed{
\begin{array}{c|c|c|c|c}
x_{11} & x_{12} & \cdots & x_{1p} & y_1 \\
\hline
x_{21} & x_{22} & \cdots & x_{2p} & y_2 \\
\hline
\vdots & \vdots & \ddots & \vdots & \vdots \\
\hline
x_{n1} & x_{n2} & \cdots & x_{np} & y_n
\end{array}
}
\end{array}
\quad\rightarrow\quad
\begin{array}{c}
\begin{array}{cccc} \,\,\mathbf{FG}  &  \mathbf{AST} & \!\!\mathbf{3PA} & \!\!\mathbf{PTS} \end{array} \\
\boxed{
\begin{array}{c|c|c|c}
1.8\,\, & \enspace0.6 \,\, & \,4.1 \, & \enspace5.3 \\
\hline
0.4\,\, & \enspace0.8 \,\, & \,1.5 \, & \enspace1.7 \\
\hline
1.1\,\, & \enspace1.9 \,\, & \,2.2 \, & \enspace3.2 \\
\hline
6.0\,\, & \enspace1.6 \,\, & \,0.0 \, & 13.9 \\
\hline
3.4\,\, & \enspace2.2 \,\, & \,0.2 \, & \enspace8.9 \\
\hline
\vdots & \vdots & \vdots & \vdots \\
\end{array}
}
\end{array}
$$

From here, we can pull out just the first row, in order to get the first observation $\{x_{11}, x_{12}, \cdots, x_{1p}, y_1\}$. Similarly, we can pull out just the second column $\{x_{12}, x_{22}, \cdots, x_{n2}\}$, in order to get the second feature.

Recall that this is our model for Multiple Linear Regression:

$$
\hat{y} = \theta_0\:+\:\theta_1x_{1}\:+\:\theta_2 x_{2}\:+\:...\:+\:\theta_p x_{p}
$$

We can then write all of our predictions $\hat{y}_i$ like this:

$$
\begin{cases}
\hat{y}_1 = \theta_0 + \theta_1 x_{11} + \theta_2 x_{12} + \cdots + \theta_p x_{1p} \\
\hat{y}_2 = \theta_0 + \theta_1 x_{21} + \theta_2 x_{22} + \cdots + \theta_p x_{2p} \\
\vdots \\
\hat{y}_n = \theta_0 + \theta_1 x_{n1} + \theta_2 x_{n2} + \cdots + \theta_p x_{np}
\end{cases}
$$

### Linear Algebra Approach

:::{tip} Linear Algebra Review: Linearity 

An expression is **linear in $\theta$** (a set of parameters) if it is a linear combination of the elements of the set. Checking if an expression can separate into a matrix product of two terms -- a **vector of $\theta$** s, and a matrix/vector **not involving $\theta$** -- is a good indicator of linearity.

For example, consider the vector $\theta = [\theta_0, \theta_1, \theta_2]$

* $\hat{y} = \theta_0 + 2\theta_1 + 3\theta_2$ is linear in theta, and we can separate it into a matrix product of two terms:

$$\hat{y} = \begin{bmatrix} 1 \space 2 \space 3 \end{bmatrix} \begin{bmatrix} \theta_0 \\ \theta_1 \\ \theta_2 \end{bmatrix}$$

* $\hat{y} = \theta_0\theta_1 + 2\theta_1^2 + 3log(\theta_2)$ is *not* linear in theta, as the $\theta_1$ term is squared, and the $\theta_2$ term is logged. We cannot separate it into a matrix product of two terms.
:::

:::{tip} Linear Algebra Review: Vector Dot Product
The **dot product (or inner product)** is a vector operation that:

* Can only be carried out on two vectors of the **same length**
* Sums up the products of the corresponding entries of the two vectors
* Returns a single number

For example, let

\begin{align}
\vec{u} = \begin{bmatrix}1 \\ 2 \\ 3\end{bmatrix}, \vec{v} = \begin{bmatrix}1 \\ 1 \\ 1\end{bmatrix}
\end{align}

The dot product between $\vec{u}$ and $\vec{v}$ is

\begin{align}
\vec{u} \cdot \vec{v} &= \vec{u}^T \vec{v} = \vec{v}^T \vec{u} \\
  &= 1 \cdot 1 + 2 \cdot 1 + 3 \cdot 1 \\ 
  &= 6
\end{align}

While not in scope, note that we can also interpret the dot product geometrically:

* It is the product of three things: the **magnitude** of both vectors, and the **cosine** of the angles between them: $$\vec{u} \cdot \vec{v} = ||\vec{u}|| \cdot ||\vec{v}|| \cdot {cos \theta}$$
:::

We now know how to generate a single prediction from multiple observed features. Data scientists usually work at scale – that is, they want to build models that can produce many predictions, all at once. The vector notation we introduced above gives us a hint on how we can expedite multiple linear regression. We want to use the tools of linear algebra.

Let's think about how we can apply what we did above. To accommodate for the fact that we're considering several feature variables, we'll adjust our notation slightly. Each observation can now be thought of as a row vector with an entry for each of $p$ features.

```{image} images/observation.png
:alt: Explanation for notation: x_i_j is the ith observation with the jth feature.
:width: 550
```

To make a prediction from the *first* observation in the data, we take the dot product of the parameter vector and *first* observation vector. To make a prediction from the *second* observation, we would repeat this process to find the dot product of the parameter vector and the *second* observation vector. If we wanted to find the model predictions for each observation in the dataset, we'd repeat this process for all $n$ observations in the data. 

$$
\begin{cases}
\hat{y}_1 = x_1^T\theta \enspace \text{,} \quad \text{where} \enspace x_1^T = \begin{bmatrix} 1 & x_{11} & x_{12} & \cdots & x_{1p} \end{bmatrix} \text{ is datapoint/observation 1}\\
\hat{y}_2 = x_2^T\theta \enspace \text{,} \quad \text{where} \enspace x_2^T = \begin{bmatrix} 1 & x_{21} & x_{22} & \cdots & x_{2p} \end{bmatrix} \text{ is datapoint/observation 2}\\
\vdots \\
\hat{y}_n = x_n^T\theta \enspace \text{,} \quad \text{where} \enspace x_n^T = \begin{bmatrix} 1 & x_{n1} & x_{n2} & \cdots & x_{np} \end{bmatrix} \text{ is datapoint/observation n}\\
\end{cases}
$$

Expanding our transpose vectors, we get the following:

$$
\hat{y}_1 = [1,\:x_{11},\:x_{12},\:x_{13},\:...,\:x_{1p}] \theta \\
\hat{y}_2 = [1,\:x_{21},\:x_{22},\:x_{23},\:...,\:x_{2p}] \theta \\
\vdots \\
\hat{y}_n = [1,\:x_{n1},\:x_{n2},\:x_{n3},\:...,\:x_{np}] \theta
$$

We can then vectorize our predictions $\hat{y}$ and our parameters $\theta$. We will keep the $n$ row vectors, each with dimension $p + 1$.

$$
\begin{bmatrix}
\hat{y}_1 \\
\\
\hat{y}_2 \\
\\
\vdots \\
\\
\hat{y}_n
\end{bmatrix}
{\Huge =} \enspace
\begin{matrix}
\begin{bmatrix} 1 & x_{11} & x_{12} & \dots & x_{1p} \end{bmatrix} \\
\\
\begin{bmatrix} 1 & x_{21} & x_{22} & \dots & x_{2p} \end{bmatrix} \\
\\
\vdots \\
\\
\begin{bmatrix} 1 & x_{n1} & x_{n2} & \dots & x_{np} \end{bmatrix}
\end{matrix}
\enspace {\Huge \theta}
$$

We can collect all of our $x_i^T$ row vectors into a single matrix.

$$
\begin{bmatrix}
\hat{y}_1 \\
\\
\hat{y}_2 \\
\\
\vdots \\
\\
\hat{y}_n
\end{bmatrix}
{\Huge =} \enspace
\begin{bmatrix}
1 & x_{11} & x_{12} & \cdots & x_{1p} \\
\\
1 & x_{21} & x_{22} & \cdots & x_{2p} \\
\vdots & \vdots & \ddots & \vdots \\
\\
1 & x_{n1} & x_{n2} & \cdots & x_{np}
\end{bmatrix}
\enspace {\Huge \theta}
$$

Briefly expanding our parameter vector $\theta$, we can see that our predictions take the following form:

$$
\begin{bmatrix}
\hat{y}_1 \\
\\
\hat{y}_2 \\
\\
\vdots \\
\\
\hat{y}_n
\end{bmatrix}
{\Huge =} \enspace
\begin{bmatrix}
1 & x_{11} & x_{12} & \cdots & x_{1p} \\
\\
1 & x_{21} & x_{22} & \cdots & x_{2p} \\
\\
\vdots & \vdots & \ddots & \vdots \\
\\
1 & x_{n1} & x_{n2} & \cdots & x_{np}
\end{bmatrix}
\enspace
\begin{bmatrix}
\theta_0 \\
\\
\theta_1 \\
\\
\theta_2 \\
\\
\vdots \\
\\
\theta_p
\end{bmatrix}
$$

Let's collapse our parameter vector back into $\theta$ and call our matrix of observations $\mathbb{X}$.

$$
\begin{bmatrix}
\hat{y}_1 \\
\\
\hat{y}_2 \\
\\
\vdots \\
\\
\hat{y}_n
\end{bmatrix}
{\Huge =} \enspace
{\Huge \mathbb{X}}
\enspace {\Huge \theta}
$$

Finally, we can call our prediction vector $\hat{\mathbb{Y}}$. We can now represent our Multiple Linear Regression model for many observations fully in terms of matrices.

$$
{\Huge \hat{\mathbb{Y}}}
{\Huge =} \enspace
{\Huge \mathbb{X}}
\enspace {\Huge \theta}
$$

The matrix $\mathbb{X}$ is known as the **design matrix**. It contains all observed data for each of our $p$ features, where each **row** corresponds to one **observation**, and each **column** corresponds to a **feature**. It often (but not always) contains an additional column of all ones to represent the **intercept** or **bias column**.

Our **true output** can also be represented as a vector, which we will call $\mathbb{R}^{p + 1}$. As an example, our true output in the basketball example was the actual recorded PTS (5.1, 1.7, etc.). This is distinct from the predictions our model would make for $PTS$ based on our features (FG, AST, 3PA)

:::{admonition} Dimension Check

It is important to keep in mind the dimension for each of the vectors and matrices we are discussing here.

* Any given observation $x_i$ has dimension $\mathbb{R}^{p + 1}$, or $x_i \in \mathbb{R}^{p + 1}$.
* Our parameter vetor $\theta$ also has dimension $\mathbb{R}^{p + 1}$, or $\theta \in \mathbb{R}^{p + 1}$.
* Our design matrix $\mathbb{X}$ has dimension $\mathbb{R}^{n \times (p + 1)}$, or $\mathbb{X} \in \mathbb{R}^{n \times (p + 1)}$.
* Our prediction vector $\hat{\mathbb{Y}}$ has dimension $\mathbb{R}^n$, or $\hat{\mathbb{Y}} \in \mathbb{R}^n$.
* Our true output vector $\mathbb{Y}$ also has dimension $\mathbb{R}^n$, or $\mathbb{Y} \in \mathbb{R}^n$.
:::

To review what is happening in the design matrix: each row represents a single observation. For example, a student in Data 100. Each column represents a feature. For example, the ages of students in Data 100. This convention allows us to easily transfer our previous work in DataFrames over to this new linear algebra perspective.

```{image} images/row_col.png
:alt: Illustration of how to view a dataframe within the linear algebra framework.
:width: 600
```

To summarize where we have ended up, our model now takes the form $\mathbb{\hat{Y}} = \mathbb{X} \theta$. $\mathbb{\hat{Y}}$ is the **prediction vector** with $n$ elements ($\mathbb{\hat{Y}} \in \mathbb{R}^{n}$); it contains the prediction made by the model for each of the $n$ input observations. $\mathbb{X}$ is the **design matrix** with dimensions $\mathbb{X} \in \mathbb{R}^{n \times (p + 1)}$, and $\theta$ is the **parameter vector** with dimensions $\theta \in \mathbb{R}^{(p + 1)}$. Our **true output** $\mathbb{Y}$ contains $n$ elements ($\mathbb{Y} \in \mathbb{R}^{n}$).

### Mean Squared Error

We now have a new approach to understanding models in terms of vectors and matrices. To accompany this new convention, we should update our understanding of risk functions and model fitting.

Recall our definition of MSE:
$$
R(\theta) = \frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2
$$

At its heart, the MSE is a measure of *distance* – it gives an indication of how "far away" the predictions are from the true values, on average. 

::: {tip} Linear Algebra: L2 Norm
When working with vectors, this idea of "distance" or the vector's **size/length** is represented by the **norm**. More precisely, the distance between two vectors $\vec{a}$ and $\vec{b}$ can be expressed as:
$$
||\vec{a} - \vec{b}||_2 = \sqrt{(a_1 - b_1)^2 + (a_2 - b_2)^2 + \ldots + (a_n - b_n)^2} = \sqrt{\sum_{i=1}^n (a_i - b_i)^2}
$$

The double bars are mathematical notation for the norm. The subscript 2 indicates that we are computing the L2, or squared norm.

The two norms we need to know for Data 100 are the L1 and L2 norms (sound familiar?). In this note, we'll focus on L2 norm. We'll dive into L1 norm in future lectures. 

For the n-dimensional vector $$\vec{x} = \begin{bmatrix} x_1 \\ x_2 \\ \vdots \\ x_n \end{bmatrix}$$ its **L2 vector norm** is

$$
||\vec{x}||_2 = \sqrt{(x_1)^2 + (x_2)^2 + \ldots + (x_n)^2} = \sqrt{\sum_{i=1}^n (x_i)^2}
$$

The L2 vector norm is a generalization of the Pythagorean theorem in $n$ dimensions. Thus, it can be used as a measure of the **length** of a vector or even as a measure of the **distance** between two vectors. 
::: 

We can express the MSE as a squared L2 norm if we rewrite it in terms of the prediction vector, $\hat{\mathbb{Y}}$, and true target vector, $\mathbb{Y}$:

$$
R(\theta) = \frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2 = \frac{1}{n} (||\mathbb{Y} - \hat{\mathbb{Y}}||_2)^2
$$

Here, the superscript 2 outside of the parentheses means that we are *squaring* the norm. If we plug in our linear model $\hat{\mathbb{Y}} = \mathbb{X} \theta$, we find the MSE cost function in vector notation:

$$
R(\theta) = \frac{1}{n} (||\mathbb{Y} - \mathbb{X} \theta||_2)^2
$$

Under the linear algebra perspective, our new task is to fit the optimal parameter vector $\theta$ such that the cost function is minimized. Equivalently, we wish to minimize the norm $$||\mathbb{Y} - \mathbb{X} \theta||_2 = ||\mathbb{Y} - \hat{\mathbb{Y}}||_2.$$ 

We can restate this goal in two ways:

* Minimize the mean squared error for the linear model $\mathbb{\hat{Y}} = \mathbb{X} \theta$
* Minimize the **distance** between the vector of true values, $\mathbb{Y}$, and the vector of predicted values, $\mathbb{\hat{Y}}$
* Minimize the **length** of the **residual vector**, defined as:
$$
e = \mathbb{Y} - \mathbb{\hat{Y}} = \begin{bmatrix}
           y_1 - \hat{y}_1 \\
           y_2 - \hat{y}_2 \\
           \vdots \\
           y_n - \hat{y}_n
         \end{bmatrix}
$$

### A Note on Terminology for Multiple Linear Regression

There are several equivalent terms in the context of regression. The ones we use most often for this course are bolded.

* $x$ can be called a
  - **Feature(s)**
  - Covariate(s)
  - **Independent variable(s)**
  - Explanatory variable(s)
  - Predictor(s)
  - Input(s)
  - Regressor(s)
* $y$ can be called an
  - **Output**
  - Outcome
  - **Response**
  - Dependent variable
* $\hat{y}$ can be called a
  - **Prediction**
  - Predicted response
  - Estimated value
* $\theta$ can be called a
  - **Weight(s)**
  - **Parameter(s)**
  - Coefficient(s)
* $\hat{\theta}$ can be called a
  - **Estimator(s)**
  - **Optimal parameter(s)**
* A datapoint $(x, y)$ is also called an observation.

## Geometric Derivation
::: {tip} Linear Algebra: Span 
Recall that the **span** or **column space** of a matrix $\mathbb{X}$ (denoted $span(\mathbb{X})$) is the set of all possible linear combinations of the matrix's columns. In other words, the span represents every point in space that could possibly be reached by adding and scaling some combination of the matrix columns. Additionally, if each column of $\mathbb{X}$ has length $n$, $span(\mathbb{X})$ is a subspace of $\mathbb{R}^{n}$.
:::

### Rethinking Matrix-Vector Multiplication
There are 2 ways we can think about matrix-vector multiplication 

1. So far, we’ve thought of our model as horizontally stacked predictions per datapoint:

```{image} images/matmul1.png
:alt: The prediction column vector Y_hat equals the product of X, made up of vertically stacked row vectors representing observations, and theta.
:width: 300
```
2. Notice that we can express our predictions $\mathbb{\hat{Y}}$ in the following way:

$$
\begin{bmatrix}
\hat{y}_1 \\
\\
\hat{y}_2 \\
\\
\vdots \\
\\
\hat{y}_n
\end{bmatrix}
{\Huge =} \enspace
\begin{bmatrix}
\theta_0 + \theta_1 x_{11} + \theta_2 x_{12} + \cdots + \theta_p x_{1p} \\
\\
\theta_0 + \theta_1 x_{21} + \theta_2 x_{22} + \cdots + \theta_p x_{2p} \\
\\
\vdots \\
\\
\theta_0 + \theta_1 x_{n1} + \theta_2 x_{n2} + \cdots + \theta_p x_{np}
\end{bmatrix}
$$

Notice that the $j$th element (zero-indexed) of every sum in the vector on the right side contains $\theta_j$. So, the zeroth element of every sum contains $\theta_0$, the second element of every sum (again, zero-indexed) contains $\theta_2$, etc. It would be nice to group these elements with shared $\theta$ values together!

Because adding vectors of the same length produces a new vector of element-wise sums, we can split this already-summed vector into the sum of vectors:

$$
\begin{bmatrix}
\hat{y}_1 \\
\\
\hat{y}_2 \\
\\
\vdots \\
\\
\hat{y}_n
\end{bmatrix}
{\Large =} \enspace
\begin{bmatrix}
\theta_0 \\
\\
\theta_0 \\
\\
\vdots \\
\\
\theta_0
\end{bmatrix}
{\Large +} \enspace
\begin{bmatrix}
\theta_1 x_{11} \\
\\
\theta_1 x_{21} \\
\\
\vdots \\
\\
\theta_2 x_{n1}
\end{bmatrix}
{\Large +} \enspace
\begin{bmatrix}
\theta_2 x_{12} \\
\\
\theta_2 x_{22} \\
\\
\vdots \\
\\
\theta_2 x_{n2}
\end{bmatrix}
{\Large +} \enspace
{\Large \cdots} \enspace
{\Large +} \enspace
\begin{bmatrix}
\theta_p x_{1p} \\
\\
\theta_p x_{2p} \\
\\
\vdots \\
\\
\theta_p x_{np}
\end{bmatrix}
$$

Note that any $j$th summed column vector contains exclusively elements of the form $\theta_j x_{ij}$ (except for the first one, which contains only elements of the form $\theta_0 = \theta_0 \times 1$). Thus, you can factor out $\theta_j$ from every $j$-th summed column vector:

$$
\begin{bmatrix}
\hat{y}_1 \\
\\
\hat{y}_2 \\
\\
\vdots \\
\\
\hat{y}_n
\end{bmatrix}
{\Large =} \enspace
{\Large \theta_0}
\begin{bmatrix}
1 \\
\\
1 \\
\\
\vdots \\
\\
1
\end{bmatrix}
{\Large +} \enspace
{\Large \theta_1}
\begin{bmatrix}
x_{11} \\
\\
x_{21} \\
\\
\vdots \\
\\
x_{n1}
\end{bmatrix}
{\Large +} \enspace
{\Large \theta_2}
\begin{bmatrix}
x_{12} \\
\\
x_{22} \\
\\
\vdots \\
\\
x_{n2}
\end{bmatrix}
{\Large +} \enspace
{\Large \cdots} \enspace
{\Large +} \enspace
{\Large \theta_p}
\begin{bmatrix}
x_{1p} \\
\\
x_{2p} \\
\\
\vdots \\
\\
x_{np}
\end{bmatrix}
$$

3. It is sometimes very helpful to think of matrix-vector multiplication as performed by columns. We can also think of $\mathbb{Y}$ as a *linear combination of feature vectors*, scaled by *parameters*.
```{image} images/matmul2.png
:alt: The prediction column vector Y_hat is the product of X and theta. This can be rewritten as the columns of X multiplied by the parameters within theta.
:width: 500
```

Up until now, we've mostly thought of our model as a scalar product between horizontally stacked observations and the parameter vector. We can also think of $\hat{\mathbb{Y}}$ as a **linear combination of feature vectors**, scaled by the **parameters**. We use the notation $\mathbb{X}_{:, i}$ to denote the $i$th column of the design matrix. 

```{image} images/columns.png
:alt: The prediction column vector Y_hat is the product of X and theta.
:width: 500
```

$$
\hat{\mathbb{Y}} = 
\theta_0 \begin{bmatrix}
           1 \\
           1 \\
           \vdots \\
           1
         \end{bmatrix} + \theta_1 \begin{bmatrix}
           x_{11} \\
           x_{21} \\
           \vdots \\
           x_{n1}
         \end{bmatrix} + \ldots + \theta_p \begin{bmatrix}
           x_{1p} \\
           x_{2p} \\
           \vdots \\
           x_{np}
         \end{bmatrix}
         = \theta_0 \mathbb{X}_{:,\:1} + \theta_1 \mathbb{X}_{:,\:2} + \ldots + \theta_p \mathbb{X}_{:,\:p+1}
$$

This new approach is useful because it allows us to take advantage of the properties of linear combinations.

Because the prediction vector, $\hat{\mathbb{Y}} = \mathbb{X} \theta$, is a **linear combination** of the columns of $\mathbb{X}$, we know that the **predictions are contained in the span of $\mathbb{X}$**. That is, we know that $\mathbb{\hat{Y}} \in \text{Span}(\mathbb{X})$.


### Column Space And The Normal Equation

The diagram below is a simplified view of $\text{Span}(\mathbb{X})$, assuming that each column of $\mathbb{X}$ has length $n$. Notice that the columns of $\mathbb{X}$ define a subspace of $\mathbb{R}^n$, where each point in the subspace can be reached by a linear combination of $\mathbb{X}$'s columns. The prediction vector $\mathbb{\hat{Y}}$ lies somewhere in this subspace.

```{image} images/span.png
:alt: Illustration of the span of X
:width: 600
```

Examining this diagram, we find a problem. The vector of true values, $\mathbb{Y}$, could theoretically lie *anywhere* in $\mathbb{R}^n$ space – its exact location depends on the data we collect out in the real world. However, our multiple linear regression model can only make predictions in the subspace of $\mathbb{R}^n$ spanned by $\mathbb{X}$. Remember the model fitting goal we established in the previous section: we want to generate predictions such that the distance between the vector of true values, $\mathbb{Y}$, and the vector of predicted values, $\mathbb{\hat{Y}}$, is minimized. This means that **we want $\mathbb{\hat{Y}}$ to be the vector in $\text{Span}(\mathbb{X})$ that is closest to $\mathbb{Y}$**. 

Another way of rephrasing this goal is to say that we wish to minimize the length of the residual vector $e$, as measured by its $L_2$ norm. 

```{image} images/residual.png
:alt: The residual is now drawn on the previous graph.
:width: 600
```

The vector in $\text{Span}(\mathbb{X})$ that is closest to $\mathbb{Y}$ is always the **orthogonal projection** of $\mathbb{Y}$ onto $\text{Span}(\mathbb{X}).$ Thus, we should choose the parameter vector $\theta$ that makes the **residual vector orthogonal to any vector in $\text{Span}(\mathbb{X})$**. You can visualize this as the vector created by dropping a perpendicular line from $\mathbb{Y}$ onto the span of $\mathbb{X}$. 

::: {tip} Linear Algebra: Orthogonality
Recall that two vectors $\vec{a}$ and $\vec{b}$ are orthogonal if their dot product is zero: $\vec{a}^{T}\vec{b} = 0$. 

A vector $v$ is **orthogonal** to the span of a matrix $M$ if and only if $v$ is orthogonal to **each column** in $M$. Put together, a vector $v$ is orthogonal to $\text{Span}(M)$ if:

$$M^Tv = \vec{0}$$

Note that $\vec{0}$ represents the **zero vector**, a $d$-length vector full of 0s.
:::

Remember our goal is to find $\hat{\theta}$ such that we minimize the objective function $R(\theta)$. Equivalently, this is the $\hat{\theta}$ such that the residual vector $e = \mathbb{Y} - \mathbb{X} \hat{\theta}$ is orthogonal to $\text{Span}(\mathbb{X})$. 

Looking at the definition of orthogonality of $\mathbb{Y} - \mathbb{X}\hat{\theta}$ to $span(\mathbb{X})$, we can write: 
$$
\mathbb{X}^T (\mathbb{Y} - \mathbb{X}\hat{\theta}) = \vec{0}
$$

Let's then rearrange the terms: 
$$
\mathbb{X}^T \mathbb{Y} - \mathbb{X}^T \mathbb{X} \hat{\theta} = \vec{0}
$$

And finally, we end up with the **normal equation**:
$$
\mathbb{X}^T \mathbb{X} \hat{\theta} = \mathbb{X}^T \mathbb{Y}
$$

Any vector $\theta$ that minimizes MSE on a dataset must satisfy this equation.

If $\mathbb{X}^T \mathbb{X}$ is invertible, we can conclude:
$$
\hat{\theta} = (\mathbb{X}^T \mathbb{X})^{-1} \mathbb{X}^T \mathbb{Y}
$$

This is called the **least squares estimate** of $\theta$: it is the value of $\theta$ that minimizes the squared loss. 

Note that the least squares estimate was derived under the assumption that $\mathbb{X}^T \mathbb{X}$ is *invertible*. This condition holds true when $\mathbb{X}^T \mathbb{X}$ is full column rank, which, in turn, happens when $\mathbb{X}$ is full column rank. We will get to the proof for why $\mathbb{X}$ needs to be full column rank in the OLS Properties section.

## OLS Properties

### Residuals
When using the optimal parameter vector, our residuals $e = \mathbb{Y} - \hat{\mathbb{Y}}$ are orthogonal to $span(\mathbb{X})$.

$$\mathbb{X}^Te = 0 $$

::: {hint} Proof

* The optimal parameter vector, $\hat{\theta}$, solves the normal equations $\implies \hat{\theta} = (\mathbb{X}^T\mathbb{X})^{-1}\mathbb{X}^T\mathbb{Y}$

$$\mathbb{X}^Te = \mathbb{X}^T (\mathbb{Y} - \mathbb{\hat{Y}}) $$

$$\mathbb{X}^T (\mathbb{Y} - \mathbb{X}\hat{\theta}) = \mathbb{X}^T\mathbb{Y} - \mathbb{X}^T\mathbb{X}\hat{\theta}$$

* Any matrix multiplied with its own inverse is the identity matrix $\mathbb{I}$

$$\mathbb{X}^T\mathbb{Y} - (\mathbb{X}^T\mathbb{X})(\mathbb{X}^T\mathbb{X})^{-1}\mathbb{X}^T\mathbb{Y} = \mathbb{X}^T\mathbb{Y} - \mathbb{X}^T\mathbb{Y} = 0$$
:::

### The Bias/Intercept Term
For all linear models with an **intercept term**, the **sum of residuals is zero**.

$$\sum_i^n e_i = 0$$

::: {tip} Proof
* For all linear models with an **intercept term**, the average of the predicted $y$ values is equal to the average of the true $y$ values.
$$\bar{y} = \bar{\hat{y}}$$
* Rewriting the sum of residuals as two separate sums,
$$\sum_i^n e_i = \sum_i^n y_i - \sum_i^n\hat{y}_i$$
* Each respective sum is a multiple of the average of the sum.
$$\sum_i^n e_i = n\bar{y} - n\bar{y} = n(\bar{y} - \bar{y}) = 0$$
:::

### Existence of a Unique Solution
To summarize:

|   | Model | Estimate | Unique? |
| -- | -- | -- |  -- | 
| Constant Model + MSE | $\hat{y} = \theta_0$| $\hat{\theta}_0 = mean(y) = \bar{y}$ | **Yes**. Any set of values has a unique mean.
| Constant Model + MAE | $\hat{y} = \theta_0$  | $\hat{\theta}_0 = median(y)$ | **Yes**, if odd. **No**, if even. Return the average of the middle 2 values.
| Simple Linear Regression + MSE | $\hat{y} = \theta_0 + \theta_1x$| $\hat{\theta}_0 = \bar{y} - \hat{\theta}_1\bar{x}$ $\hat{\theta}_1 = r\frac{\sigma_y}{\sigma_x}$| **Yes**. Any set of non-constant* values has a unique mean, SD, and correlation coefficient.
| **OLS** (Linear Model + MSE) | $\mathbb{\hat{Y}} = \mathbb{X}\mathbb{\theta}$| $\hat{\theta} = (\mathbb{X}^T\mathbb{X})^{-1}\mathbb{X}^T\mathbb{Y}$  | **Yes**, if $\mathbb{X}$ is full column rank (all columns are linearly independent, # of datapoints >>> # of features).

### Uniqueness of the OLS Solution

In most settings, the number of observations (n) is much greater than the number of features (p).

```{image} images/solution_matrices.png
:alt: Conceptual illustration of the normal equation with matrices.
:width: 500
```

In practice, instead of directly inverting matrices, we can use more efficient numerical solvers to directly solve a system of linear equations using the normal equation shown below. Note that at least one solution always exists because intuitively, we can always draw a line of best fit for a given set of data, but there may be multiple lines that are “equally good”. (Formal proof is beyond this course.)

```{image} images/normal_equation.png
:alt: Conceptual illustration of the normal equation with matrices.
:width: 400
```

The Least Squares estimate $\hat{\theta}$ is **unique** if and only if $\mathbb{X}$ is **full column rank**.

::: {hint} Proof

* We know the solution to the normal equation $\mathbb{X}^T\mathbb{X}\hat{\theta} = \mathbb{X}^T\mathbb{Y}$ is the least square estimate that minimizes the squared loss.
* $\hat{\theta}$ has a **unique** solution $\iff$ the square matrix $\mathbb{X}^T\mathbb{X}$ is **invertible** $\iff$ $\mathbb{X}^T\mathbb{X}$ is full rank.
  * The **column rank** of a square matrix is the max number of linearly independent columns it contains.
  * An $n$ x $n$ square matrix is deemed full column rank when all of its columns are linearly independent. That is, its rank would be equal to $n$.
  * $\mathbb{X}^T\mathbb{X}$ has shape $(p + 1) \times (p + 1)$, and therefore has max rank $p + 1$. 
* $rank(\mathbb{X}^T\mathbb{X})$ = $rank(\mathbb{X})$ (proof out of scope).
* Therefore, $\mathbb{X}^T\mathbb{X}$ has rank $p + 1$ $\iff$  $\mathbb{X}$ has rank $p + 1$ $\iff \mathbb{X}$ is full column rank.
:::

Therefore, if $\mathbb{X}$ is not full column rank, we will not have unique estimates. This can happen for two major reasons.

1. If our design matrix $\mathbb{X}$ is "**wide**":
    * If n < p, then we have way more features (columns) than observations (rows).
    * Then $rank(\mathbb{X}) = min(n, p+1) < p+1$, so $\hat{\theta}$ is not unique.
    * <u>Intuition</u>
      * With one data point ($n=1$), there are $\infty$ SLR models ($p = 1$) that minimize MSE. Try drawing a picture!
    * <u>More intuition</u>
      * Given two non-overlapping vectors ($p = 1$) in 2D space ($n = 2$) there is exactly one linear combination that can reach each point in 2D space.
      * If you have three of these vectors ($p = 2$), there are $\infty$ possible combinations.
    * Typically we have $n$ >> $p$, so this is less of an issue.

2. If our design matrix $\mathbb{X}$ has features that are **linear combinations** of other features:
    * By definition, rank of $\mathbb{X}$ is number of linearly independent columns in $\mathbb{X}$.
    * Example: If “Width”, “Height”, and “Perimeter” are all columns,
      * Perimeter = 2 * Width + 2 * Height  $\rightarrow$  $\mathbb{X}$ is not full rank.
    * Important with one-hot encoding (to discuss later).

## Evaluating Model Performance

Our geometric view of multiple linear regression has taken us far! We have identified the optimal set of parameter values to minimize MSE in a model of multiple features. Now, we want to understand how well our fitted model performs. 

### RMSE
One measure of model performance is the **Root Mean Squared Error**, or RMSE. The RMSE is simply the square root of MSE. Taking the square root converts the value back into the original, non-squared units of $y_i$, which is useful for understanding the model's performance. A low RMSE indicates more "accurate" predictions – that there is a lower average loss across the dataset.

$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2}$$

::: {note}

Note that RMSE is more interpretable as it is in the same units as the original data. On the other hand, MSE is differentiable everywhere making it useful for running optimization algorithms like gradient descent.

:::


### Multiple $R^2$
For SLR, we used the correlation coefficient to capture the association between the target variable and a single feature variable. In a multiple linear model setting, we will need a performance metric that can account for multiple features at once. **Multiple $R^2$**, also called the **coefficient of determination**, is the **proportion of variance** of our **fitted values** (predictions) $\hat{y}_i$ to our true values $y_i$. It ranges from 0 to 1 and is effectively the *proportion* of variance in the observations that the **model explains**.

$$R^2 = \frac{\text{variance of } \hat{y}_i}{\text{variance of } y_i} = \frac{\sigma^2_{\hat{y}}}{\sigma^2_y}$$

Note that for OLS with an intercept term, for example $\hat{y} = \theta_0 + \theta_1x_1 + \theta_2x_2 + \cdots + \theta_px_p$, $R^2$ is equal to the square of the correlation between $y$ and $\hat{y}$. On the other hand for SLR, $R^2$ is equal to $r^2$, the correlation between $x$ and $y$. The proof of these last two properties is out of scope for this course.

As we add more features, our fitted values tend to become closer and closer to our actual values. Thus, $R^2$ increases.

Returning to our basketball example:
* The SLR model (AST only) explains **45.7%** of the variance in the true $y$.
* The AST & 3PA model explains **60.9%** of the variance in the true $y$.

Adding more features doesn't always mean our model is better, though! We'll see why later in the course.


## Demo: Predicting Penguin Bill Depth

Can we predict bill depth from flipper length and body mass? We can now apply what we have learned about Multiple Linear Regression models and Ordinary Least Squares to do just that. We will conclude by introducing the  `scikit-learn` library for simple machine learning tasks, and using it to recreate the same prediction of bill depth.

Consider the `penguins` dataset.

In [42]:
# Load in the penguins dataset provided by seaborn package
df = pl.from_pandas(sns.load_dataset("penguins"))

# Filter to only include Adelie species and remove any rows with missing values.
# Remember that dropping rows with missing values should be done CAREFULLY.
# Here, we drop rows just so the demo modeling code runs without error.
df = df.filter(pl.col("species") == "Adelie").drop_nulls()

df

species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
str,str,f64,f64,f64,f64,str
"""Adelie""","""Torgersen""",39.1,18.7,181.0,3750.0,"""Male"""
"""Adelie""","""Torgersen""",39.5,17.4,186.0,3800.0,"""Female"""
"""Adelie""","""Torgersen""",40.3,18.0,195.0,3250.0,"""Female"""
"""Adelie""","""Torgersen""",36.7,19.3,193.0,3450.0,"""Female"""
"""Adelie""","""Torgersen""",39.3,20.6,190.0,3650.0,"""Male"""
…,…,…,…,…,…,…
"""Adelie""","""Dream""",36.6,18.4,184.0,3475.0,"""Female"""
"""Adelie""","""Dream""",36.0,17.8,195.0,3450.0,"""Female"""
"""Adelie""","""Dream""",37.8,18.1,193.0,3750.0,"""Male"""


Suppose we could measure flippers and weight easily, but not bill dimensions.
How can we predict **bill depth** from flipper length and/or body mass?

```{image} images/penguin.png
:alt: Profile illustration of a penguin. The vertical depth of the penguin's bill, measured at its base, is labeled as bill depth.
:width: 400
```

For demo purposes, we'll drop all columns except the variables whose relationships we're interested in modeling.

In [43]:
df = df.select("bill_depth_mm", "flipper_length_mm", "body_mass_g")

df

bill_depth_mm,flipper_length_mm,body_mass_g
f64,f64,f64
18.7,181.0,3750.0
17.4,186.0,3800.0
18.0,195.0,3250.0
19.3,193.0,3450.0
20.6,190.0,3650.0
…,…,…
18.4,184.0,3475.0
17.8,195.0,3450.0
18.1,193.0,3750.0


Suppose we want to create a linear regression model that predicts a penguin's **bill depth** $y$ using both their **flipper length** $x_1$ and **body mass** $x_2$, plus an intercept term.

$$
\Large \hat{y} = \theta_0 + \theta_1 x_1 + \theta_2 x_2
$$

Another way to write this:

$$
\widehat{\texttt{bill\_depth\_mm}} = \theta_0 + \theta_1 * \texttt{flipper\_length\_mm} + \theta_2 * \texttt{body\_mass\_g}
$$

### Using The Solution To The Normal Equation

We saw in last lecture that we can model the above multiple linear regression equation using matrix multiplication:

$$
\large \hat{\mathbb{Y}} = \mathbb{X} \theta
$$

The optimal $\hat{\theta}$ that minimizes MSE also solves the **normal equation**:

$$
\left( \mathbb{X}^T \mathbb{X} \right) \hat{\theta} = \mathbb{X}^T \mathbb{Y}
$$

If $\mathbb{X}$ is full column rank, then there is a unique solution to $\hat{\theta}$:

$$
\large \hat{\theta} = \left( \mathbb{X}^T \mathbb{X} \right)^{-1} \mathbb{X}^T \mathbb{Y}
$$

> Note: This derivation is one of the most challenging in the course, especially for those of you who are learning linear algebra during the same semester. Be kind to yourself!

Let's try this in code. We have to add a bias column to the design matrix if we want to include an intercept in the model:

In [44]:
# Construct the design matrix X as a dataframe and add a bias column
X = df.select("flipper_length_mm", "body_mass_g")

# Notice how using 1 (or pl.lit(1)) broadcasts a single value to all rows.
X = X.with_columns(bias=1)

# Let's put the bias column in front.
X = X.select("bias", "flipper_length_mm", "body_mass_g")

X

bias,flipper_length_mm,body_mass_g
i32,f64,f64
1,181.0,3750.0
1,186.0,3800.0
1,195.0,3250.0
1,193.0,3450.0
1,190.0,3650.0
…,…,…
1,184.0,3475.0
1,195.0,3450.0
1,193.0,3750.0


In [45]:
# Construct the outcome vector y
y = df["bill_depth_mm"]

y

bill_depth_mm
f64
18.7
17.4
18.0
19.3
20.6
…
18.4
17.8
18.1


Recall the solution to the normal equation if $\mathbb{X}$ is full column rank:

$$
\hat{\theta} = \left( \mathbb{X}^T \mathbb{X} \right)^{-1} \mathbb{X}^T \mathbb{Y}
$$

In [46]:
# .to_numpy() hands the dataframe and the outcome vector to NumPy as matrices
# M.T transposes a matrix M
# X @ Y multiplies matrices X and Y
# np.linalg.inv(M) finds the inverse of a matrix M
X_mat, y_vec = X.to_numpy(), y.to_numpy()
theta_using_normal_equation = np.linalg.inv(X_mat.T @ X_mat) @ X_mat.T @ y_vec

theta_using_normal_equation

array([1.10029953e+01, 9.82848689e-03, 1.47749591e-03])

So, our fitted model is as follows:

$$ \widehat{\texttt{bill\_depth\_mm}} = 11 + 0.0098 * \texttt{flipper\_length\_mm} + 0.0015 * \texttt{body\_mass\_g} $$

Note: The code above is inefficient. We won't go into this in Data 100, but it's better to use `np.linalg.solve` rather than computing the explicit matrix inverse:

In [47]:
# Solves for θ in (XTX)θ = XTy (i.e., the normal equation) using a system of linear equations
np.linalg.solve(X_mat.T @ X_mat, X_mat.T @ y_vec)

array([1.10029953e+01, 9.82848689e-03, 1.47749591e-03])

#### Make Predictions
We'll save the predictions in a column so we can compare them against predictions from `sklearn` in the next section.

In [48]:
# Convert the dataframe representation of the design matrix to a numpy array, 
# and then multiply it by the theta vector to get the predicted outcomes.
# Yhat = Xθ
df = df.with_columns(
    pred_bill_depth_mm=pl.Series(X.to_numpy() @ theta_using_normal_equation)
)

df

bill_depth_mm,flipper_length_mm,body_mass_g,pred_bill_depth_mm
f64,f64,f64,f64
18.7,181.0,3750.0,18.322561
17.4,186.0,3800.0,18.445578
18.0,195.0,3250.0,17.721412
19.3,193.0,3450.0,17.997254
20.6,190.0,3650.0,18.263268
…,…,…,…
18.4,184.0,3475.0,17.945735
17.8,195.0,3450.0,18.016911
18.1,193.0,3750.0,18.440503


### Machine Learning Frameworks

We have introduced the Multiple Linear Regression model $\hat{\mathbb{Y}} = \mathbb{X}\theta$. We have expressed  the Mean Square Error loss function in vector notation:

$$
R(\theta) = \frac{1}{n} (||\mathbb{Y} - \mathbb{X} \theta||_2)^2
$$

We then used geometry to minimize average loss by **fitting** our model with optimal parameter values $\hat{\theta}$. We touched upon the use of matrix calculus to fit the model (out of scope for this course), and we used calculus to find the optimal paramaeter values for SLR in the previous chapters.

In research and industry, it is common to use libraries for creating and training models. In Data 100, we will use `scikit-learn`, commonly called `sklearn`.

```{image} images/sklearn.png
:alt: Home page for the scikit-learn website. Three sections are featured: Classification, Regression, and Clustering.
:width: 500
```

:::{tip} Other ML Libraries

`scikit-learn` has many great tutorials and is the standard for simple ML tasks.

Some other machine learning libraries include:

* XGboost
* Pytorch
* TensorFlow
* Hugging Face
:::

#### The `sklearn` Workflow

At a high level, there are three steps to creating an `sklearn` model:

1. **Initialize** a new model instance.
  * *Make a “copy” of the model template*

```python
my_model = lm.LinearRegression()
```

2. **Fit** the model to the training data.
  * *Save the optimal model parameters*

```python
my_model.fit(X, y)
```

3. Use fitted models to make **predictions.**

```python
my_model.predict(X_new)
```

To extract the fitted model's **parameters**, use `my_model.coef_` (to extract parameters for all features without the bias $\theta_1, \theta_2, \cdots, \theta_p$) and `my_model.intercept_` (to extract the parameter for the bias $\theta_0$).

Let's try this out.

### Using `sklearn` to fit our Multiple Linear Regression Model

Do we obtain the same fitted OLS parameters provided by the normal equation using `sklearn`? We can use the `sklearn` workflow we just described to find out.

An alternate approach to optimize our model is to use the `sklearn.linear_model.LinearRegression` class.[(Documentation)](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)

In [49]:
# Import the LinearRegression class
from sklearn.linear_model import LinearRegression

1. **Create an `sklearn` object.**

    First we create a model. At this point the model has not been fit so it has no parameters.

In [50]:
# Construct a blank LinearRegression model object
model = LinearRegression()

model

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


2. **`fit` the object to data.**

Then we "fit" the model, which means computing the parameters that minimize the loss function. 
    
The first argument of the fit function should be a matrix (or `DataFrame`) with dimension $n \times p + 1$, and the second argument should be a vector (or `Series`) of the observations we're trying to predict. 

Note: The `LinearRegression` class is hard-coded to use the **MSE** as its loss function.

In [51]:
# Fit the linear regression model to the data
model.fit(
    X=df.select("flipper_length_mm", "body_mass_g"),
    y=df["bill_depth_mm"]
  )

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](2,)","[0.01,0. ]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](2,)","['flipper_length_mm','body_mass_g']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,11
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,2
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(2)


3. **Analyze fit or call `predict`.**

Now that our model is trained, we can ask it questions.

The code below asks the model to estimate the bill depth (in mm) for three different penguins with different flipper lengths and body masses.

In [52]:
# Suppose we collected data from three new penguins:
new_penguins = pl.DataFrame({
    'flipper_length_mm': [185, 175, 145],
    'body_mass_g': [3750.0, 3550.0, 3250.0],
})
# Generate a prediction (yhat) by providing the X values to the model.

model.predict(new_penguins)

array([18.36187501, 17.96809096, 17.22998758])

Here are the predictions for all of the penguins in our dataset:

In [53]:
model.predict(df.select("flipper_length_mm", "body_mass_g"))

array([18.32256106, 18.44557829, 17.72141192, 17.99725413, 18.26326785,
       18.13787407, 19.82684359, 17.5197668 , 18.49472072, 19.45001768,
       18.28800021, 18.0169111 , 19.58793878, 17.72411076, 19.11520455,
       17.73663808, 18.09110819, 18.47506375, 18.65737419, 18.38660737,
       18.45540678, 18.04671885, 17.56890923, 17.34760713, 18.60823175,
       17.55432765, 18.51469999, 17.72648731, 18.57367091, 17.83222412,
       19.06098673, 18.70651662, 18.01723339, 17.6576879 , 19.68179284,
       17.445892  , 18.68178426, 17.41133115, 19.4303607 , 17.25375308,
       19.66688896, 17.85220338, 17.96776867, 19.01184429, 18.00232952,
       19.20398322, 17.96776867, 18.95255108, 17.12566046, 18.34697113,
       18.07620431, 18.5143777 , 16.99281474, 18.45033139, 17.47537746,
       19.42053222, 18.14025062, 18.87392319, 17.0223002 , 18.7261736 ,
       17.86916151, 18.90848404, 17.37677031, 19.52389247, 18.18939306,
       18.63264183, 18.17448918, 19.07081521, 18.33714265, 19.19

Let's save the `scikit-learn` model predictions as a column in our dataframe:

In [54]:
# Can also provide a design matrix or dataframe to .predict()
df = df.with_columns(
    sklearn_preds=pl.Series(model.predict(df[["flipper_length_mm", "body_mass_g"]]))
)

df

bill_depth_mm,flipper_length_mm,body_mass_g,pred_bill_depth_mm,sklearn_preds
f64,f64,f64,f64,f64
18.7,181.0,3750.0,18.322561,18.322561
17.4,186.0,3800.0,18.445578,18.445578
18.0,195.0,3250.0,17.721412,17.721412
19.3,193.0,3450.0,17.997254,17.997254
20.6,190.0,3650.0,18.263268,18.263268
…,…,…,…,…
18.4,184.0,3475.0,17.945735,17.945735
17.8,195.0,3450.0,18.016911,18.016911
18.1,193.0,3750.0,18.440503,18.440503


Looking at the predictions above, we see that they are exactly the same as what we calculated with our analytic formula!

**Analyze parameters.**

We can ask the model for its intercept and slope with `_intercept` and `_coef`, respectively.

In [55]:
# There is one intercept term in the model
# Why is this a scalar?
model.intercept_

np.float64(11.002995277447068)

In [56]:
# There are two slope terms in the model
# Why is this an array?
model.coef_

array([0.00982849, 0.0014775 ])

So, our fitted model is as follows:

$$
\widehat{\texttt{bill\_depth\_mm}} = 11 + 0.0098 * \texttt{flipper\_length\_mm} + 0.0015 * \texttt{body\_mass\_g}
$$

The parameters are the same as with our analytic formula.

In [57]:
# vs. analytical solutions
theta_using_normal_equation

array([1.10029953e+01, 9.82848689e-03, 1.47749591e-03])

They look the same!

## Fitting General Models

The solution we found for the normal equation can only help us *fit* our model, or find our optimal parameters $\hat{\theta}$ when we use  Multiple Linear Regression as our model and Mean Squared Error as our loss function. How do these software packages, like `scikit-learn`, fit general models?

We made some big assumptions with the calculus-based and geometric techniques for minimizing loss:

* **Calculus:**
  * We can't always algebraically solve for the zero points of the derivative.
* **Geometric:**
  * We don't always use a linear model with MSE loss.

To design more complex models with different loss functions, we need a new optimization technique: **gradient descent**.

In the next chapter, we will use this **iterative algorithm** to numerically compute the paramaters $\hat{\theta}$ that **minime the loss**.
